# Import path and libraries

In [1]:
# ============================================================
# CELL 1: DWT + PCA + SVM - IMPORTS AND CONFIGURATION
# ============================================================
#
# Purpose:
#     Configure paths and import libraries required for the
#     DWT + PCA + SVM classification pipeline.
#
# Pipeline:
#
#     Tumor ROI
#          ↓
#     DWT Feature Extraction
#          ↓
#     20 DWT Features
#          ↓
#     StandardScaler
#          ↓
#     PCA
#          ↓
#     15 PCA Components
#          ↓
#     SVM
#
# IMPORTANT:
# - PCA has already been performed.
# - This notebook does NOT perform PCA again.
# - The DWT PCA train/test CSV files are used directly.
# - The existing MODELS folder is reused.
# - A DWT-specific model filename will be used so the
#   existing GLCM SVM model is not overwritten.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.svm import SVC

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize


# ============================================================
# 2. PATH CONFIGURATION
# ============================================================

FEATURE_DIR = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT"
)


# ============================================================
# 3. DWT + PCA DATASET FILES
# ============================================================

PCA_TRAIN_FILE = (
    FEATURE_DIR /
    "dwt_pca_train.csv"
)

PCA_TEST_FILE = (
    FEATURE_DIR /
    "dwt_pca_test.csv"
)


# ============================================================
# 4. MODEL OUTPUT DIRECTORY
# ============================================================

MODEL_DIR = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\MODELS"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. DISPLAY CONFIGURATION
# ============================================================

print("=" * 70)
print("DWT + PCA + SVM CLASSIFICATION")
print("=" * 70)


print("\nDWT + PCA Train file:")
print(PCA_TRAIN_FILE)


print("\nDWT + PCA Test file:")
print(PCA_TEST_FILE)


print("\nModel directory:")
print(MODEL_DIR)


print("\n" + "=" * 70)
print("CELL 1 COMPLETE")
print("=" * 70)

DWT + PCA + SVM CLASSIFICATION

DWT + PCA Train file:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_pca_train.csv

DWT + PCA Test file:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features_DWT\dwt_pca_test.csv

Model directory:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\MODELS

CELL 1 COMPLETE


# Load pca

In [2]:
# ============================================================
# CELL 2: LOAD DWT + PCA DATA
# ============================================================

print("=" * 70)
print("LOADING DWT + PCA DATA")
print("=" * 70)


# ------------------------------------------------------------
# Load CSV files
# ------------------------------------------------------------

train_df = pd.read_csv(
    PCA_TRAIN_FILE
)

test_df = pd.read_csv(
    PCA_TEST_FILE
)


# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

print(
    "\nTrain shape:",
    train_df.shape
)

print(
    "Test shape :",
    test_df.shape
)


# ------------------------------------------------------------
# Check class column
# ------------------------------------------------------------

if "class" not in train_df.columns:

    raise ValueError(
        "'class' column not found in training CSV."
    )


if "class" not in test_df.columns:

    raise ValueError(
        "'class' column not found in testing CSV."
    )


# ------------------------------------------------------------
# Separate labels
# ------------------------------------------------------------

y_train = train_df["class"].astype(str)

y_test = test_df["class"].astype(str)


# ------------------------------------------------------------
# Extract numeric PCA features
# ------------------------------------------------------------

X_train = (
    train_df
    .drop(columns=["class"])
    .select_dtypes(include=[np.number])
)


X_test = (
    test_df
    .drop(columns=["class"])
    .select_dtypes(include=[np.number])
)


# ------------------------------------------------------------
# Remove accidental index columns
# ------------------------------------------------------------

X_train = X_train.loc[
    :,
    ~X_train.columns.str.startswith("Unnamed")
]


X_test = X_test.loc[
    :,
    ~X_test.columns.str.startswith("Unnamed")
]


# ------------------------------------------------------------
# Convert to float32
# ------------------------------------------------------------

X_train = X_train.astype(
    np.float32
)

X_test = X_test.astype(
    np.float32
)


# ============================================================
# DISPLAY INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DWT + PCA FEATURE INFORMATION")
print("=" * 70)


print(
    "\nNumber of PCA features:",
    X_train.shape[1]
)


print(
    "Training samples:",
    X_train.shape[0]
)


print(
    "Testing samples :",
    X_test.shape[0]
)


print("\nPCA feature columns:")

print(
    X_train.columns.tolist()
)


# ============================================================
# CHECK TRAIN / TEST FEATURE MATCH
# ============================================================

if list(X_train.columns) != list(X_test.columns):

    raise ValueError(
        "Train and test PCA feature columns do not match."
    )


# ============================================================
# CHECK NaN / INFINITE VALUES
# ============================================================

if not np.isfinite(
    X_train.to_numpy()
).all():

    raise ValueError(
        "NaN or infinite values found in training data."
    )


if not np.isfinite(
    X_test.to_numpy()
).all():

    raise ValueError(
        "NaN or infinite values found in testing data."
    )


print(
    "\nNo NaN or infinite values found."
)


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TRAIN CLASS DISTRIBUTION")
print("=" * 70)

print(
    y_train.value_counts()
)


print("\n" + "=" * 70)
print("TEST CLASS DISTRIBUTION")
print("=" * 70)

print(
    y_test.value_counts()
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("DWT + PCA DATA READY FOR SVM")
print("=" * 70)

print(
    "\n✓ DWT + PCA training data loaded"
)

print(
    "✓ DWT + PCA testing data loaded"
)

print(
    f"✓ PCA components: {X_train.shape[1]}"
)

print(
    "✓ Train/test feature columns match"
)

print(
    "✓ No NaN or infinite values found"
)

print(
    "\nReady for SVM training."
)

LOADING DWT + PCA DATA

Train shape: (3883, 17)
Test shape : (830, 17)

DWT + PCA FEATURE INFORMATION

Number of PCA features: 15
Training samples: 3883
Testing samples : 830

PCA feature columns:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15']

No NaN or infinite values found.

TRAIN CLASS DISTRIBUTION
class
pituitary     1455
meningioma    1320
glioma        1108
Name: count, dtype: int64

TEST CLASS DISTRIBUTION
class
meningioma    301
pituitary     295
glioma        234
Name: count, dtype: int64

DWT + PCA DATA READY FOR SVM

✓ DWT + PCA training data loaded
✓ DWT + PCA testing data loaded
✓ PCA components: 15
✓ Train/test feature columns match
✓ No NaN or infinite values found

Ready for SVM training.


# SVM Hypertunning 

In [3]:
# ============================================================
# CELL 3: FAST DWT + PCA + SVM HYPERPARAMETER TUNING
# ============================================================
#
# Purpose:
#     Find the best SVM hyperparameters using ONLY:
#
#         X_train
#         y_train
#
# Pipeline:
#
#     DWT Features
#          ↓
#     PCA (15 components)
#          ↓
#     SVM Hyperparameter Tuning
#
# IMPORTANT:
#     X_test and y_test are NOT used in this cell.
#
# SPEED OPTIMIZATION:
#     - Small parameter grid
#     - 3-fold Stratified Cross-Validation
#     - probability=False during tuning
#     - n_jobs=-1
#
# Total combinations:
#
#     Linear SVM = 2
#     RBF SVM    = 2
#
# Total = 4 combinations
#
# With 3-fold CV:
#
#     4 × 3 = 12 model fits
# ============================================================


from sklearn.svm import SVC

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    ParameterGrid
)


print("=" * 70)
print("FAST DWT + PCA + SVM HYPERPARAMETER TUNING")
print("=" * 70)


# ============================================================
# 1. BASE SVM MODEL
# ============================================================

svm = SVC(
    probability=False,
    random_state=42
)


# ============================================================
# 2. SMALL PARAMETER GRID
# ============================================================

param_grid = [

    # --------------------------------------------------------
    # LINEAR KERNEL
    # --------------------------------------------------------

    {
        "kernel": ["linear"],

        "C": [
            1,
            10
        ]
    },


    # --------------------------------------------------------
    # RBF KERNEL
    # --------------------------------------------------------

    {
        "kernel": ["rbf"],

        "C": [
            1,
            10
        ],

        "gamma": [
            "scale"
        ]
    }
]


# ============================================================
# 3. COUNT PARAMETER COMBINATIONS
# ============================================================

number_of_combinations = len(
    list(
        ParameterGrid(param_grid)
    )
)


print(
    "\nNumber of parameter combinations:",
    number_of_combinations
)


# ============================================================
# 4. STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


print(
    "Cross-validation folds:",
    cv.n_splits
)


print(
    "Total model fits:",
    number_of_combinations * cv.n_splits
)


# ============================================================
# 5. GRID SEARCH
# ============================================================
#
# scoring="f1_macro":
#
# Gives equal importance to all three tumor classes.
#
# This is useful because your classes are not perfectly
# balanced.
# ============================================================

grid_search = GridSearchCV(

    estimator=svm,

    param_grid=param_grid,

    cv=cv,

    scoring="f1_macro",

    n_jobs=-1,

    verbose=2,

    return_train_score=False,

    refit=False
)


# ============================================================
# 6. DISPLAY TRAINING INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("STARTING DWT + PCA + SVM HYPERPARAMETER SEARCH")
print("=" * 70)


print(
    "\nTraining samples:",
    X_train.shape[0]
)


print(
    "PCA features:",
    X_train.shape[1]
)


print(
    "\nOnly X_train and y_train are being used."
)


print(
    "X_test and y_test are NOT used."
)


# ============================================================
# 7. RUN GRID SEARCH
# ============================================================

grid_search.fit(
    X_train,
    y_train
)


# ============================================================
# 8. GET BEST PARAMETERS
# ============================================================

best_params = grid_search.best_params_


# ============================================================
# 9. GET BEST CROSS-VALIDATION SCORE
# ============================================================

best_cv_score = grid_search.best_score_


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BEST DWT + PCA + SVM PARAMETERS")
print("=" * 70)


print(
    "\nBest Parameters:"
)


print(
    best_params
)


print(
    "\nBest Cross-Validation Macro-F1:"
)


print(
    f"{best_cv_score * 100:.2f}%"
)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DWT + PCA + SVM HYPERPARAMETER TUNING COMPLETE")
print("=" * 70)


print(
    "\nBest Parameters:",
    best_params
)


print(
    "Best CV Macro-F1:",
    f"{best_cv_score * 100:.2f}%"
)


print(
    "\nThese parameters will be used to train the final"
)


print(
    "DWT + PCA + SVM model in the next cell."
)

FAST DWT + PCA + SVM HYPERPARAMETER TUNING

Number of parameter combinations: 4
Cross-validation folds: 3
Total model fits: 12

STARTING DWT + PCA + SVM HYPERPARAMETER SEARCH

Training samples: 3883
PCA features: 15

Only X_train and y_train are being used.
X_test and y_test are NOT used.
Fitting 3 folds for each of 4 candidates, totalling 12 fits

BEST DWT + PCA + SVM PARAMETERS

Best Parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Best Cross-Validation Macro-F1:
74.79%

DWT + PCA + SVM HYPERPARAMETER TUNING COMPLETE

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV Macro-F1: 74.79%

These parameters will be used to train the final
DWT + PCA + SVM model in the next cell.


# Train and save svm model 

In [4]:
# ============================================================
# CELL 4: TRAIN AND SAVE FINAL DWT + PCA + SVM
# ============================================================
#
# Purpose:
#     Train the final SVM using the best hyperparameters
#     obtained from Cell 3.
#
# Pipeline:
#
#     DWT → PCA (15 components) → SVM
#
# IMPORTANT:
#     - Only X_train and y_train are used for training.
#     - X_test and y_test are NOT used here.
#     - probability=True is enabled for ROC/AUC evaluation.
#     - Existing models are NOT overwritten.
# ============================================================


print("=" * 70)
print("TRAINING FINAL DWT + PCA + SVM")
print("=" * 70)


# ============================================================
# 1. BEST PARAMETERS FROM CELL 3
# ============================================================

print(
    "\nBest SVM parameters:"
)

print(
    best_params
)


# ============================================================
# 2. CREATE FINAL SVM
# ============================================================

final_svm = SVC(
    **best_params,
    probability=True,
    random_state=42
)


# ============================================================
# 3. TRAIN FINAL SVM
# ============================================================

print("\nTraining final DWT + PCA + SVM...")

final_svm.fit(
    X_train,
    y_train
)


# ============================================================
# 4. DISPLAY TRAINING INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DWT + PCA + SVM TRAINING COMPLETE")
print("=" * 70)


print(
    "\nTraining samples:",
    X_train.shape[0]
)


print(
    "PCA features:",
    X_train.shape[1]
)


print(
    "Kernel:",
    final_svm.kernel
)


print(
    "C:",
    final_svm.C
)


print(
    "Gamma:",
    final_svm.gamma
)


print(
    "Probability enabled:",
    final_svm.probability
)


# ============================================================
# 5. SAVE FINAL DWT + PCA + SVM MODEL
# ============================================================
#
# IMPORTANT:
#     Use a DWT-specific filename so the existing GLCM SVM
#     model is NOT overwritten.
# ============================================================

SVM_MODEL_FILE = (
    MODEL_DIR /
    "final_svm_dwt_pca.pkl"
)


joblib.dump(
    final_svm,
    SVM_MODEL_FILE
)


# ============================================================
# 6. VERIFY SAVED MODEL
# ============================================================

print(
    "\nSaved DWT + PCA + SVM model:"
)

print(
    SVM_MODEL_FILE
)


print(
    "\nFinal DWT + PCA + SVM model saved successfully."
)


# ============================================================
# 7. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("CELL 4 COMPLETE")
print("=" * 70)


print(
    "\nPipeline:",
    "DWT → PCA → SVM"
)


print(
    "PCA components:",
    X_train.shape[1]
)


print(
    "Best parameters:",
    best_params
)


print(
    "Model file:",
    SVM_MODEL_FILE
)


print(
    "\nThe saved model will be loaded in the next cell"
)


print(
    "for test prediction and probability prediction."
)

TRAINING FINAL DWT + PCA + SVM

Best SVM parameters:
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

Training final DWT + PCA + SVM...

FINAL DWT + PCA + SVM TRAINING COMPLETE

Training samples: 3883
PCA features: 15
Kernel: rbf
C: 10
Gamma: scale
Probability enabled: True

Saved DWT + PCA + SVM model:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\MODELS\final_svm_dwt_pca.pkl

Final DWT + PCA + SVM model saved successfully.

CELL 4 COMPLETE

Pipeline: DWT → PCA → SVM
PCA components: 15
Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Model file: C:\Users\harsh\OneDrive\Documents\BE_Major_Project\MODELS\final_svm_dwt_pca.pkl

The saved model will be loaded in the next cell
for test prediction and probability prediction.


# SVM testing

In [5]:
# ============================================================
# CELL 5: LOAD SAVED DWT + PCA SVM + TEST PREDICTION
# ============================================================

print("=" * 70)
print("LOADING SAVED DWT + PCA SVM AND PREDICTING TEST DATA")
print("=" * 70)


# ============================================================
# 1. LOAD SAVED SVM MODEL
# ============================================================

print(
    "\nLoading saved DWT + PCA SVM model..."
)

svm_model = joblib.load(
    SVM_MODEL_FILE
)

print(
    "DWT + PCA SVM model loaded successfully."
)

print(
    "\nModel:"
)

print(
    svm_model
)


# ============================================================
# 2. VERIFY TEST DATA
# ============================================================

print("\n" + "=" * 70)
print("TEST DATA")
print("=" * 70)

print(
    "\nTest samples:",
    X_test.shape[0]
)

print(
    "PCA features:",
    X_test.shape[1]
)

print(
    "Classes:",
    svm_model.classes_
)


# ============================================================
# 3. CHECK FEATURE COUNT
# ============================================================

if X_test.shape[1] != svm_model.n_features_in_:

    raise ValueError(
        "Test feature count does not match the features "
        "used to train the SVM."
    )


print(
    "\nFeature count verified."
)


# ============================================================
# 4. PREDICT TEST CLASSES
# ============================================================

print("\n" + "=" * 70)
print("GENERATING CLASS PREDICTIONS")
print("=" * 70)

y_pred = svm_model.predict(
    X_test
)


print(
    "\nClass predictions generated."
)

print(
    "Prediction shape:",
    y_pred.shape
)


# ============================================================
# 5. PREDICT TEST PROBABILITIES
# ============================================================

print("\n" + "=" * 70)
print("GENERATING CLASS PROBABILITIES")
print("=" * 70)

y_proba = svm_model.predict_proba(
    X_test
)


print(
    "\nClass probabilities generated."
)

print(
    "Probability shape:",
    y_proba.shape
)


# ============================================================
# 6. GET PREDICTION CONFIDENCE
# ============================================================

prediction_confidence = np.max(
    y_proba,
    axis=1
)


# ============================================================
# 7. DISPLAY SAMPLE PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE PREDICTIONS")
print("=" * 70)

print(
    "\nFirst 10 actual labels:"
)

print(
    y_test.iloc[:10].to_numpy()
)


print(
    "\nFirst 10 predicted labels:"
)

print(
    y_pred[:10]
)


print(
    "\nFirst 10 probability predictions:"
)

print(
    y_proba[:10]
)


print(
    "\nFirst 10 prediction confidences:"
)

print(
    prediction_confidence[:10]
)


# ============================================================
# 8. VERIFY PROBABILITY VALUES
# ============================================================

probability_sums = np.sum(
    y_proba,
    axis=1
)


if not np.allclose(
    probability_sums,
    1.0
):

    raise ValueError(
        "Invalid probability values detected."
    )


print(
    "\nProbability values verified."
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DWT + PCA SVM TEST PREDICTION COMPLETE")
print("=" * 70)

print(
    "\nTest samples:",
    len(y_pred)
)

print(
    "Number of classes:",
    len(svm_model.classes_)
)

print(
    "Prediction shape:",
    y_pred.shape
)

print(
    "Probability shape:",
    y_proba.shape
)

print(
    "\nClass order:"
)

print(
    svm_model.classes_
)

print(
    "\n✓ Saved DWT + PCA SVM loaded"
)

print(
    "✓ Test predictions generated"
)

print(
    "✓ Test probabilities generated"
)

print(
    "✓ Probability values verified"
)

print(
    "\nReady for DWT + PCA SVM evaluation."
)

LOADING SAVED DWT + PCA SVM AND PREDICTING TEST DATA

Loading saved DWT + PCA SVM model...
DWT + PCA SVM model loaded successfully.

Model:
SVC(C=10, probability=True, random_state=42)

TEST DATA

Test samples: 830
PCA features: 15
Classes: ['glioma' 'meningioma' 'pituitary']

Feature count verified.

GENERATING CLASS PREDICTIONS

Class predictions generated.
Prediction shape: (830,)

GENERATING CLASS PROBABILITIES

Class probabilities generated.
Probability shape: (830, 3)

SAMPLE PREDICTIONS

First 10 actual labels:
['glioma' 'glioma' 'glioma' 'glioma' 'glioma' 'glioma' 'glioma' 'glioma'
 'glioma' 'glioma']

First 10 predicted labels:
['glioma' 'pituitary' 'glioma' 'glioma' 'glioma' 'pituitary' 'glioma'
 'glioma' 'glioma' 'glioma']

First 10 probability predictions:
[[0.98270973 0.01025066 0.00703961]
 [0.25595996 0.21259592 0.53144412]
 [0.51933884 0.10725204 0.37340912]
 [0.9676176  0.02461477 0.00776763]
 [0.96262552 0.02333603 0.01403844]
 [0.22035376 0.03664161 0.74300463]
 [0.9

# SVM Evauluation 

In [6]:
# ============================================================
# CELL 6: DWT + PCA + SVM CLASSIFICATION EVALUATION
# ============================================================
#
# Purpose:
#     Evaluate the final DWT + PCA + SVM model on the
#     unseen test dataset.
#
# Metrics:
#     - Accuracy
#     - Precision
#     - Recall
#     - F1-score
#     - Classification Report
#
# IMPORTANT:
#     - The model is NOT retrained in this cell.
#     - y_pred was generated in Cell 5.
#     - y_test contains the actual test labels.
#     - PCA is NOT performed again.
#     - DWT is NOT performed again.
# ============================================================


print("=" * 70)
print("DWT + PCA + SVM CLASSIFICATION EVALUATION")
print("=" * 70)


# ============================================================
# 1. ACCURACY
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)


# ============================================================
# 2. PRECISION
# ============================================================

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


# ============================================================
# 3. RECALL
# ============================================================

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


# ============================================================
# 4. F1-SCORE
# ============================================================

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


# ============================================================
# 5. DISPLAY OVERALL METRICS
# ============================================================

print("\n" + "=" * 70)
print("OVERALL TEST METRICS")
print("=" * 70)

print(
    f"\nAccuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)


# ============================================================
# 6. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# 7. SAVE METRICS
# ============================================================

evaluation_results = {

    "feature_extractor": "DWT",

    "pca_components": X_train.shape[1],

    "classifier": "SVM",

    "accuracy": accuracy,

    "macro_precision": precision,

    "macro_recall": recall,

    "macro_f1": f1
}


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DWT + PCA + SVM EVALUATION COMPLETE")
print("=" * 70)

print(
    f"\nAccuracy  : {accuracy * 100:.2f}%"
)

print(
    f"Precision : {precision * 100:.2f}%"
)

print(
    f"Recall    : {recall * 100:.2f}%"
)

print(
    f"F1-Score  : {f1 * 100:.2f}%"
)

print(
    "\nPCA components:",
    X_test.shape[1]
)

print(
    "\nEvaluation results stored in:"
)

print(
    "evaluation_results"
)

DWT + PCA + SVM CLASSIFICATION EVALUATION

OVERALL TEST METRICS

Accuracy  : 76.39%
Precision : 76.97%
Recall    : 75.73%
F1-Score  : 76.11%

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      glioma       0.80      0.68      0.73       234
  meningioma       0.82      0.83      0.83       301
   pituitary       0.69      0.76      0.72       295

    accuracy                           0.76       830
   macro avg       0.77      0.76      0.76       830
weighted avg       0.77      0.76      0.76       830


DWT + PCA + SVM EVALUATION COMPLETE

Accuracy  : 76.39%
Precision : 76.97%
Recall    : 75.73%
F1-Score  : 76.11%

PCA components: 15

Evaluation results stored in:
evaluation_results
